<a href="https://colab.research.google.com/github/kimgayeon430/skala-LangChain/blob/main/LangChain_Reservation_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗓️ AI 예약 도우미 챗봇 (LangChain + Gradio)

## 프로젝트 개요
사용자가 자연어로 예약 문의를 하면, AI가 날짜·시간·인원 등 필요한 정보를 스스로 파악하고
누락된 정보는 되물어서 채운 뒤 예약을 확정해주는 대화형 챗봇입니다.
기존의 버튼/메뉴 선택식 예약 시스템과 달리, 자유로운 문장 그대로 입력해도
필요한 정보를 정확히 추출해 처리하는 것이 핵심입니다.

## 서비스 모델: B2B2C
본 챗봇은 식당·미용실 등 소상공인(B)이 자사 홈페이지나 카카오톡 채널에 임베드하여,
최종 고객(C)이 별도 앱 설치나 회원가입 없이 대화만으로 예약할 수 있도록 제공하는
**B2B2C 구조**를 전제로 설계되었습니다. 소상공인은 이 챗봇을 자신의 서비스에 붙여
고객 응대를 자동화하고, 고객은 편한 대화형 인터페이스로 예약 경험을 얻습니다.

## 주요 기능
- 자연어 문장에서 예약 정보(날짜, 시간, 인원 등)를 구조화된 데이터로 자동 추출
- 정보가 부족할 경우 필요한 항목만 골라서 되묻는 멀티턴 대화
- 대화 흐름 속에서 지금까지 파악된 정보를 누적하여 최종 예약 확정

## 사용 기술
- **LangChain** — `init_chat_model`, `ChatPromptTemplate`, `with_structured_output`
- **Pydantic** — 예약 정보 스키마 정의 및 유효성 검증
- **Gradio** — 대화형 챗봇 UI (`ChatInterface`)

## 업종별 확장성
예약 항목(스키마)만 업종에 맞게 바꾸면 다양한 소상공인에게 재사용할 수 있습니다.
예: 미용실 → 담당 디자이너 지정, 식당 → 좌석 유형, 병원 → 진료 과목 등.

## 식당 예약 규칙
**대상**
- 단일 식당 예약

**필수 정보**
- 이름
- 연락처
- 날짜
- 시간
- 인원수

**선택 정보**
- 요청사항 (창가 자리, 알러지 등 — 사용자가 먼저 말한 경우에만 반영)

**영업 규칙**
- 영업시간: 11:00 ~ 22:00
- 휴무 없음
- 최대 수용 인원: 9명
- 10명 이상 예약 요청 시 → 자동 예약 대신 전화 문의 안내

**대화 방식**
- 사용자가 필요한 정보를 한 번에 다 줘도, 하나씩 나눠서 줘도 응답 가능해야 함
- 필수 정보가 모두 모이기 전에는 예약을 확정하지 않음
- "내일", "모레", "다음주 금요일" 같은 상대적 날짜 표현은 오늘 날짜를 기준으로 계산

**예약 완료 시**
- 입력받은 예약 정보를 요약해서 보여줌

## Package 설치

In [1]:
!pip install langchain-openai langchain_google_genai langchain_tavily

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 17.3 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled googl

## API Key

In [2]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path="/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")
print(".env 내 GOOGLE_API_KEY가 환경변수에 할당됐습니다:", os.environ["GOOGLE_API_KEY"][:5]+"*****")
#print(".env 내 TAVILY_API_KEY가 환경변수에 할당됐습니다:", os.environ["TAVILY_API_KEY"][:5]+"*****")

.env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다: sk-pr*****
.env 내 GOOGLE_API_KEY가 환경변수에 할당됐습니다: AQ.Ab*****


## 오늘 날짜 준비
'내일/모레'와 같은 표현 사용하기 위함

In [5]:
from datetime import datetime
today_date = datetime.now()
print(today_date)

2026-09-15 00:34:06.891584


## 예약 처리 Tool 함수 작성

In [66]:
from langchain_core.tools import tool

# 예약 목록을 저장발 리스트
reservations = []

# 영업 규칙 지정용 상수
OPEN_TIME = "11:00"
CLOSE_TIME = "22:00"
MAX_CAPACITY = 9

@tool
def make_reservation(name: str, phone: str, date: str, time: str, party: int, request: str="") -> str:
  """식당 예약을 접수한다.
  date는 YYYY-MM-DD형태, time은 HH:MM형태로 반환한다.
  예약 가능 여부는 이 도구를 실행해야만 확인할 수 있다.""" # 예약 가능 여부를 tool 호출 없이 llm이 판단해서 출력하는 문제가 발생해 해당 조건 추가함

  if party > MAX_CAPACITY :
    return "10명 이상의 예약은 별도 전화 문의 부탁드립니다."
  if not(OPEN_TIME <= time <= CLOSE_TIME) :
    return "저희 식당은 11시 오픈, 22시 마감입니다. 영업시간의 예약만 예약이 가능합니다."
  # 여기까지 왔다면 예약 가능한 상태
  reservation_info = {
      "name": name,
      "phone": phone,
      "date": date,
      "time": time,
      "party": party,
      "request": request
  }
  reservations.append(reservation_info)
  return f"예약이 접수되었습니다. \n < 예약 정보 > \n 이름 : {name}님 \n 전화번호 : {phone} \n 날짜 : {date} \n 시간 : {time} \n 인원수 : {party} \n 요청사항 : {request} "

In [67]:
# 날짜 형식 수정
today_str = today_date.strftime("%Y-%m-%d")
print(today_str)

2026-09-15


### System Prompt 만들기

In [68]:
system_prompt = f"""
당신은 식당 예약을 돕는 agent 입니다.
사용자는 편리하게 자연어로 예약하고 싶은 정보를 입력하고, 당신이 그 정보를 알맞게 추출하여 식당에게 전달합니다.
오늘은 {today_str}입니다. 사용자가 내일, 오늘 등의 상대적인 날짜를 말하면 오늘 날짜를 기준으로 계산해서 날짜 형식으로 응답하세요.
에약에 필요한 정보 : 이름, 전화번호, 날짜, 시간, 인원수, 요청사항
요청사항은 필수 정보가 아니니 사용자가 입력하지 않는다면 빈 문자열로 비워두고, 따로 요청한다면 추가하세요.
필수 정보가 다 입력되지 않았다면 사용자에게 물어보세요.
필수 정보다 다 모이기 전까진 make_reservation tool을 사용하지 마세요.
반대로 필수 정보가 모두 모였다면, 예약 가능 여부를 당신이 짐작하지 마세요.
반드시 make_reservation tool을 사용하고, 도구가 반환한 메시지를 사용자에게 전달하세요.
"""

## 모델 준비

In [69]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai",
    temperature=0.7
)

## Agent 구성

In [70]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[make_reservation],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

## Agent 테스트

### test1 : 정상 예약

In [71]:
# 정상 예약
message = {"messages" : [{"role": "user", "content": "홍길동, 010-1234-5678, 내일 저녁 7시에 4명 예약할게요"}]}
config = {"configurable" : {"thread_id" : "test1"}}

result = agent.invoke(message, config)

print(result["messages"][-1].content)

예약이 성공적으로 접수되었습니다.

- 이름: 홍길동님
- 전화번호: 010-1234-5678
- 날짜: 2026-09-16
- 시간: 19:00
- 인원수: 4명
- 요청사항: 없음

감사합니다! 추가로 도움이 필요하시면 언제든지 말씀해 주세요.


### Test2 : 인원수 오류

In [84]:
message = {"messages" : [{"role": "user", "content": "홍길동, 010-1234-5678, 내일 저녁 7시에 15명 예약할게요"}]}
config = {"configurable" : {"thread_id" : "test2"}}

result = agent.invoke(message, config)

print(result["messages"][-1].content)

예약 진행 결과, 10명 이상의 예약은 별도로 전화 문의가 필요하다고 합니다. 전화로 직접 문의하시길 권장드립니다. 다른 도움이 필요하시면 말씀해 주세요!


### Test 3 : 영업시간 이외 시간 예약

In [76]:
#멀티턴 테스트
message = {"messages" : [{"role": "user", "content": "홍길동, 010-1234-5678, 내일 저녁 11시에 4명 예약할게요"}]}
config = {"configurable" : {"thread_id" : "test3"}}

result = agent.invoke(message, config)

print(result["messages"][-1].content)

예약을 시도했으나, 저희 식당은 11시에 오픈하고 22시에 마감합니다. 따라서 23시에 예약이 불가능합니다. 

다른 시간을 원하시거나 추가로 도움이 필요하시면 말씀해 주세요!


### Test 4 : "다음주 토요일" 이라는 상대적 날짜 사용

In [78]:
message = {"messages" : [{"role": "user", "content": "홍길동, 010-1234-5678, 다음주 토요일 저녁 7시에 4명 예약할게요"}]}
config = {"configurable" : {"thread_id" : "test4"}}

result = agent.invoke(message, config)

print(result["messages"][-1].content)

예약이 성공적으로 접수되었습니다. 

**예약 정보:**
- 이름: 홍길동님
- 전화번호: 010-1234-5678
- 날짜: 2026-09-23
- 시간: 19:00
- 인원수: 4명
- 요청사항: 없음

이용해 주셔서 감사합니다! 추가로 도움이 필요하시면 언제든지 말씀해 주세요.


### Test 5 : 요청사항 추가

In [80]:
message = {"messages" : [{"role": "user", "content": "홍길동, 010-1234-5678, 내일 저녁 7시에 4명 예약할게요. 테라스자리로 부탁해요."}]}
config = {"configurable" : {"thread_id" : "test5"}}

result = agent.invoke(message, config)

print(result["messages"][-1].content)

예약이 성공적으로 접수되었습니다. 아래는 예약 정보입니다:

- 이름: 홍길동님
- 전화번호: 010-1234-5678
- 날짜: 2026-09-16
- 시간: 19:00
- 인원수: 4명
- 요청사항: 테라스자리

감사합니다! 예약에 대해 추가로 궁금한 점이 있으시면 언제든지 말씀해 주세요.


### Test 6 : 멀티턴 - 1차에서 정보 누락 요청 후 2차에서 나머지 정보 제공 -> 정상적으로 예약 처리됨

In [81]:
# test6-1 : 정보 누락 케이스
message = {"messages" : [{"role": "user", "content": "홍길동, 010-1234-5678"}]}
config = {"configurable" : {"thread_id" : "test6"}}

result = agent.invoke(message, config)

print(result["messages"][-1].content)

예약을 도와드리겠습니다. 다음 정보를 추가로 요청드립니다.

1. 예약 날짜 (예: 내일, 2026-09-16)
2. 예약 시간 (예: 19:00)
3. 인원수 (예: 2명)
4. 요청사항 (선택 사항)

이 정보를 알려주시면 예약을 진행하겠습니다!


In [82]:
# test6-2 :멀티턴 검증 완료
message = {"messages" : [{"role": "user", "content": "내일 저녁 7시에 4명 예약할게요. 테라스자리로 부탁해요."}]}
config = {"configurable" : {"thread_id" : "test6"}}  # 멀티턴 테스트를 진행하기 위해 같은 위 코드와 같은 thread_id를 사용했음

result = agent.invoke(message, config)

print(result["messages"][-1].content)

예약이 성공적으로 접수되었습니다! 

아래는 예약 정보입니다:
- 이름: 홍길동님
- 전화번호: 010-1234-5678
- 날짜: 2026-09-16
- 시간: 19:00
- 인원수: 4명
- 요청사항: 테라스자리

즐거운 저녁 되시길 바랍니다! 추가로 도움이 필요하시면 언제든지 말씀해 주세요.
